# Week 6 ChronoPDE — corrected smoke gate

This smoke-only notebook tests whether the boundary-aware DCT model can overfit four trajectories. It runs the corrected strict gate for at most 10,000 optimizer steps, reports progress once per minute, packages the diagnostic, and does **not** launch full training.

Attach the private dataset containing `chronopde.h5`, enable a T4 GPU and Internet, then use **Save Version → Save & Run All**.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import hashlib
import json
import shutil
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

import torch
IMPLEMENTATION_COMMIT = '094d0e4ce0063da22dab511b54cc6a7c576a4169'
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Enable GPU in Kaggle Notebook settings'
print('GPU:', torch.cuda.get_device_name(0))

## Verify the frozen dataset

In [ ]:
KAGGLE_INPUT = Path('/kaggle/input')
EXPECTED_DATA_SIZE = 2_389_261_328
EXPECTED_DATA_SHA256 = '907aa0d79e604e68ce2d4f5cccfd93ffc64eb68c473caf3edae4be438472caec'
data_candidates = [p for p in KAGGLE_INPUT.rglob('chronopde.h5') if p.is_file() and p.stat().st_size == EXPECTED_DATA_SIZE]
assert data_candidates, 'Attach the private dataset containing chronopde.h5'
DATA = data_candidates[0]
digest = hashlib.sha256()
with DATA.open('rb') as stream:
    for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
        digest.update(block)
assert digest.hexdigest() == EXPECTED_DATA_SHA256, 'Dataset checksum mismatch'
print('Verified dataset:', DATA)

## Install the corrected implementation and initialize writable state

In [ ]:
REPOSITORY = Path('/kaggle/working/Chrono_pde')
subprocess.run(['git', 'clone', 'https://github.com/madhavkapoor13/ChronoPDE.git', str(REPOSITORY)], check=True)
subprocess.run(['git', '-C', str(REPOSITORY), 'fetch', 'origin', IMPLEMENTATION_COMMIT], check=True)
subprocess.run(['git', '-C', str(REPOSITORY), 'checkout', '--detach', IMPLEMENTATION_COMMIT], check=True)
os.chdir(REPOSITORY)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--ignore-requires-python', '-e', '.'], check=True)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == IMPLEMENTATION_COMMIT

STATE_ROOT = Path('/kaggle/working/chronopde_smoke_state')
WRITABLE_RUNS = STATE_ROOT / 'runs'
WRITABLE_RUNS.mkdir(parents=True, exist_ok=True)
LOCAL_RUNS = REPOSITORY / 'artifacts/runs'
LOCAL_RUNS.parent.mkdir(parents=True, exist_ok=True)
if LOCAL_RUNS.exists() or LOCAL_RUNS.is_symlink():
    if LOCAL_RUNS.is_symlink() or LOCAL_RUNS.is_file(): LOCAL_RUNS.unlink()
    elif not any(LOCAL_RUNS.iterdir()): LOCAL_RUNS.rmdir()
    else: raise RuntimeError(f'Cannot safely replace non-empty {LOCAL_RUNS}')
LOCAL_RUNS.symlink_to(WRITABLE_RUNS, target_is_directory=True)
print('Writable runs:', LOCAL_RUNS.resolve())

## Run and monitor the strict four-trajectory gate

In [ ]:
SMOKE_RUN = WRITABLE_RUNS / 'chronopde-full-train-s0-smoke'
METRICS = SMOKE_RUN / 'metrics.jsonl'
SUMMARY = SMOKE_RUN / 'summary.json'
command = [sys.executable, 'scripts/train.py', '--config', 'configs/project.yaml', '--model', 'chronopde', '--regime', 'full', '--seed', '0', '--device', 'cuda', '--data-path', str(DATA), '--smoke-overfit']
process = subprocess.Popen(command)
previous_count = -1
while process.poll() is None:
    rows = [json.loads(line) for line in METRICS.read_text().splitlines() if line.strip()] if METRICS.is_file() else []
    if len(rows) != previous_count:
        latest = rows[-1] if rows else None
        print(datetime.now().isoformat(timespec='seconds'), 'smoke epochs:', len(rows), 'optimizer steps:', None if latest is None else int(latest['optimizer_steps']), 'latest:', latest)
        previous_count = len(rows)
    time.sleep(60)
process.wait()
assert SUMMARY.is_file(), 'Smoke run ended without a summary'
summary = json.loads(SUMMARY.read_text())
print(json.dumps(summary, indent=2))
if summary.get('passed') is True:
    print('STRICT SMOKE GATE PASSED')
else:
    print('STRICT SMOKE GATE FAILED — do not start full training')

## Package the diagnostic

In [ ]:
from IPython.display import FileLink, display
package = shutil.make_archive('/kaggle/working/chronopde_week6_smoke_result', 'zip', root_dir=WRITABLE_RUNS, base_dir='chronopde-full-train-s0-smoke')
print('Smoke diagnostic:', package)
display(FileLink(package))